In [1]:
import os
import pickle
import random
import cs336_basics.BytePairEncoding as BPE
TS_vocab = pickle.load(open("../model/bpe_tinystories_vocab.pkl", "rb"))
TS_merges = pickle.load(open("../model/bpe_tinystories_merges.pkl", "rb"))
OWT_vocab = pickle.load(open("../model/bpe_openwebtext_vocab.pkl", "rb"))
OWT_merges = pickle.load(open("../model/bpe_openwebtext_merges.pkl", "rb"))
DELIM = b"<|endoftext|>"

In [2]:
def sample_story(path, delim=DELIM, window=8192):
    filesize = os.path.getsize(path)

    with open(path, "rb") as f:
        # pick random location
        pos = random.randint(0, filesize - 1)
        f.seek(pos)

        # search backwards for previous delimiter
        start = pos
        while start > 0:
            step = min(window, start)
            f.seek(start - step)
            chunk = f.read(step)

            idx = chunk.rfind(delim)
            if idx != -1:
                start = start - step + idx + len(delim)
                break

            start -= step
        else:
            start = 0

        # search forwards for next delimiter
        f.seek(pos)
        end = pos

        while True:
            chunk = f.read(window)
            if not chunk:
                end = filesize
                break

            idx = chunk.find(delim)
            if idx != -1:
                end = end + idx
                break

            end += len(chunk)

        # read the story
        f.seek(start)
        story = f.read(end - start)

    return story.decode("utf-8", errors="ignore")

In [3]:
TS_stories = []
OWT_stories = []
for _ in range(10):
    TS_stories.append(sample_story("../data/TinyStoriesV2-GPT4-valid.txt"))
    OWT_stories.append(sample_story("../data/owt_valid.txt"))

In [4]:
TS_tokenizer = BPE.BPETokenizer.from_files(
    vocab_filepath="../model/bpe_tinystories_vocab.pkl",
    merges_filepath="../model/bpe_tinystories_merges.pkl",
    special_tokens=["<|endoftext|>"],
)
OWT_tokenizer = BPE.BPETokenizer.from_files(
    vocab_filepath="../model/bpe_openwebtext_vocab.pkl",
    merges_filepath="../model/bpe_openwebtext_merges.pkl",
    special_tokens=["<|endoftext|>"],
)

In [5]:
for story in TS_stories:
    tokens = TS_tokenizer.encode(story)
    print(f"Compression ratio for TinyStories story: {len(story.encode('utf-8')) / len(tokens):.4f}")

Compression ratio for TinyStories story: 3.9553
Compression ratio for TinyStories story: 3.3798
Compression ratio for TinyStories story: 4.1327
Compression ratio for TinyStories story: 4.3284
Compression ratio for TinyStories story: 3.9065
Compression ratio for TinyStories story: 4.3349
Compression ratio for TinyStories story: 4.4605
Compression ratio for TinyStories story: 3.8836
Compression ratio for TinyStories story: 4.0559
Compression ratio for TinyStories story: 4.5066


In [6]:
for story in OWT_stories:
    tokens = OWT_tokenizer.encode(story)
    print(f"Compression ratio for OpenWebText story: {len(story.encode('utf-8')) / len(tokens):.4f}")

Compression ratio for OpenWebText story: 4.7777
Compression ratio for OpenWebText story: 4.2161
Compression ratio for OpenWebText story: 3.4948
Compression ratio for OpenWebText story: 4.6209
Compression ratio for OpenWebText story: 4.5948
Compression ratio for OpenWebText story: 4.2881
Compression ratio for OpenWebText story: 4.0399
Compression ratio for OpenWebText story: 4.4815
Compression ratio for OpenWebText story: 4.3718
Compression ratio for OpenWebText story: 3.7269


In [7]:
for story in OWT_stories:
    tokens = TS_tokenizer.encode(story)
    print(f"Compression ratio for OpenWebText story with TS_tokenizer: {len(story.encode('utf-8')) / len(tokens):.4f}")

Compression ratio for OpenWebText story with TS_tokenizer: 3.0944
Compression ratio for OpenWebText story with TS_tokenizer: 2.9473
Compression ratio for OpenWebText story with TS_tokenizer: 3.0362
Compression ratio for OpenWebText story with TS_tokenizer: 3.3268
Compression ratio for OpenWebText story with TS_tokenizer: 3.0670
Compression ratio for OpenWebText story with TS_tokenizer: 3.5698
Compression ratio for OpenWebText story with TS_tokenizer: 3.4391
Compression ratio for OpenWebText story with TS_tokenizer: 3.7761
Compression ratio for OpenWebText story with TS_tokenizer: 3.4510
Compression ratio for OpenWebText story with TS_tokenizer: 2.8691


In [ ]:
with open("../data/TinyStoriesV2-GPT4-valid.txt", "r") as f:
    TS_full_tokens = TS_tokenizer.encode_iterable(f)

In [11]:
len(TS_full_tokens)

5465963